# Arriendos Gran Concepción — Portal Inmobiliario

Descarga, revisión de calidad y procesamiento de avisos de arriendo (casas y departamentos).

## 1. Setup

In [80]:
import sys
sys.executable

'/Users/ianlopezcandia/.pyenv/versions/3.10.0/bin/python'

In [82]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import plotly.express as px
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import COMMUNES, TYPES
from src.portal.crawler import run_crawl, RAW_DIR
from src.geocoding import update_cache
from src.uf import get_uf

PROCESSED_DIR = ROOT / "data" / "processed"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Descarga
Elegir comunas y número máximo de páginas. Las búsquedas ya descargadas hoy se saltan.
**Si solo quieres analizar datos ya descargados, sáltate esta sección.**

In [54]:
# Test run
# communes_to_run = {k: COMMUNES[k] for k in ["Concepción", "Talcahuano"]}
# MAX_PAGES = 3

# Full run
communes_to_run = COMMUNES
MAX_PAGES = 50

In [55]:
with ThreadPoolExecutor(max_workers=1) as executor:
    run_dir = executor.submit(run_crawl, communes=communes_to_run, max_pages=MAX_PAGES).result()

print("Guardados en:", run_dir)

Concepción - departamento
    page 1: 48 listings
    page 2: 48 listings
    page 3: 48 listings
    page 4: 48 listings
    page 5: 48 listings
    page 6: 48 listings
    page 7: 48 listings
    page 8: 48 listings
    page 9: 48 listings
    page 10: 48 listings
    page 11: 48 listings
    page 12: 48 listings
    page 13: 48 listings
    page 14: 48 listings
    page 15: 48 listings
    page 16: 48 listings
    page 17: 48 listings
    page 18: 48 listings
    page 19: 48 listings
    page 20: 48 listings
    page 21: 20 listings
  -> saved 980 listings

Concepción - casa
    page 1: 48 listings
    page 2: 40 listings
  -> saved 88 listings

Talcahuano - departamento
    page 1: 48 listings
    page 2: 6 listings
  -> saved 54 listings

Talcahuano - casa
    page 1: 29 listings
  -> saved 29 listings

Hualpén - departamento
    page 1: 31 listings
  -> saved 31 listings

Hualpén - casa
    page 1: 2 listings
  -> saved 2 listings

San Pedro de la Paz - departamento
    page 1: 4

## 3. Carga

In [56]:
# Latest run, or set a specific date: RUN = "2026-10-24"
RUN = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())[-1]
run_dir = RAW_DIR / RUN

files = sorted(run_dir.glob("*.csv"))
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(f"Run {RUN}: {len(files)} files, {len(df)} listings")

Run 2026-10-24: 16 files, 1635 listings


In [57]:
df.pivot_table(index="commune", columns="property_type",
               values="id", aggfunc="count", fill_value=0)

property_type,casa,departamento
commune,,
Chiguayante,23,58
Concepción,88,980
Coronel,13,4
Hualpén,2,31
Hualqui,2,0
Penco,1,0
San Pedro de la Paz,60,278
Talcahuano,29,54
Tomé,7,5


## 4. Calidad

In [58]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
})
quality[quality["missing"] > 0].sort_values("missing_%", ascending=False)

,missing,missing_%
total_m2,1617,98.9
broker,1404,85.9
usable_m2,53,3.2
bedrooms,17,1.0
price,10,0.6
bathrooms,8,0.5
neighborhood,7,0.4


## 5. Procesamiento
1. Unir y quitar duplicados
2. Filtrar avisos inválidos
3. Agregar coordenadas (solo se consultan direcciones nuevas; el resto sale del caché)
4. Guardar en `data/processed/`

In [101]:
# 5.1  Unir, filtrar y convertir UF a pesos
MAX_UF_RENT = 300      # above this it's a sale price (~$12M per month)
MAX_BEDROOMS = 10      # above this it's not a dwelling (offices, hostel, institute)

consolidated = (
    df.drop_duplicates(subset=["id"])
      .drop(columns=["total_m2", "broker"], errors="ignore")
      .dropna(subset=["price"])
)

# Filter invalid listings
is_sale_price = (consolidated["currency"] == "UF") & (consolidated["price"] > MAX_UF_RENT)
not_dwelling = consolidated["bedrooms"] > MAX_BEDROOMS
excluded = consolidated[is_sale_price | not_dwelling]
consolidated = consolidated[~(is_sale_price | not_dwelling)].copy()

print(f"Rows: {len(df)} -> {len(consolidated)} ({len(excluded)} excluded)")
display(excluded[["property_type", "commune", "price", "currency", "bedrooms", "title"]])

# Convert UF to CLP using the UF value of each download date
uf_by_date = {d: get_uf(d) for d in consolidated["scraped_at"].unique()}
consolidated["uf_value"] = consolidated["scraped_at"].map(uf_by_date)
consolidated["price_clp"] = (
    consolidated["price"]
    .where(consolidated["currency"] == "CLP",
           consolidated["price"] * consolidated["uf_value"])
    .round(0)
)

Rows: 1635 -> 1621 (4 excluded)


,property_type,commune,price,currency,bedrooms,title
38,casa,Concepción,7000000.0,CLP,20.0,Casa Remodelada 16 Estacionamientos
44,casa,Concepción,7600.0,UF,5.0,Excelente Casa Condominio Lomas De San Andrés
348,departamento,Concepción,4450.0,UF,2.0,Departamento Con Vista Al Rio Bio Bio
1020,departamento,Concepción,2800.0,UF,2.0,"Central, A Pasos De Paicaví Y Uss, 2 Dorm Mari..."


In [102]:
# 5.3a  PRUEBA: geolocalizar solo 30 direcciones (ejecutar una vez y revisar)
cache = update_cache(consolidated.head(30))
cache[["location", "precision", "lat", "lon"]].tail(10)

In cache: 1169 | new to geocode: 0 (about 0 min)


,location,precision,lat,lon
1159,"JAIME REPULLO, TALCAHUANO, Carriel, Talcahuano",street,-36.754471,-73.084826
1160,"Calle J 811, Talcahuano, Chile, Brisas Del Sol...",address,-36.770810,-73.067272
1161,"La Vega 801, Talcahuano, Talcahuano",address,-36.779425,-73.080797
1162,"ALTO HORNO, SECTOR LAS HIGUERAS., Talcahuano",street,-36.739713,-73.108705
1163,"C. I 832, Concepción, Talcahuano, Brisas Del S...",address,-36.770532,-73.070428
1164,"Av. Ifarle Ote. 6117, Talcahuano, Bío Bío, Chi...",address,-36.775240,-73.074702
1165,"La Vega 597, Carriel, Talcahuano",address,-36.779425,-73.080797
1166,"Av. Pingueral, Tomé, Pingueral, Tomé",street,-36.529223,-72.926995
1167,"Werner, 2500, Ciudad De Tomé, Tomé",street,-36.605335,-72.969575
1168,"Av. Werner 2500, Tome, Tomé, Ciudad De Tomé, Tomé",address,-36.609562,-72.964808


In [61]:
## borra el cache de la geolocalización para que se vuelva a intentar la geolocalización de las direcciones que no se encontraron
##from src.geocoding import forget_not_found
##forget_not_found()

In [103]:
# 5.3b  Geolocalizar todo (la primera vez tarda 20-40 min)
cache = update_cache(consolidated)
cache["precision"].value_counts()

In cache: 1169 | new to geocode: 0 (about 0 min)


precision
address         596
street          342
neighborhood    183
not_found        48
Name: count, dtype: int64

In [104]:
# 5.4  Unir coordenadas y guardar
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed = consolidated.merge(
    cache[["location", "lat", "lon", "precision","matched_address"]],
    on="location", how="left",
)

output_file = PROCESSED_DIR / f"portal_{RUN}.csv"
processed.to_csv(output_file, index=False, encoding="utf-8")

print(f"{len(processed)} listings, {processed['lat'].notna().sum()} with coordinates")
print("Saved:", output_file.relative_to(ROOT))

1621 listings, 1564 with coordinates
Saved: data/processed/portal_2026-10-24.csv


In [105]:
# 5.5  Distribución de precios en pesos (verificación)
processed.groupby("property_type")["price_clp"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).round(0)

,count,mean,std,min,1%,5%,50%,95%,99%,max
property_type,,,,,,,,,,
casa,221.0,1307837.0,805430.0,350000.0,352000.0,470000.0,1100000.0,3000000.0,3667154.0,4945256.0
departamento,1400.0,556140.0,189505.0,190000.0,320000.0,360000.0,520000.0,850000.0,1350500.0,1950000.0


In [106]:
# 5.6  Coordenadas por comuna y precisión
summary = (
    processed.assign(precision=processed["precision"].fillna("not_found"))
    .pivot_table(index="commune", columns="precision",
                 values="id", aggfunc="count", fill_value=0)
)
summary["%_with_coords"] = (
    100 * (1 - summary.get("not_found", 0) / summary.sum(axis=1))
).round(1)
summary

precision,address,neighborhood,not_found,street,%_with_coords
commune,,,,,
Chiguayante,41,20,1,19,98.8
Concepción,596,133,17,312,98.4
Coronel,2,4,4,7,76.5
Hualpén,12,4,10,7,69.7
Hualqui,0,0,2,0,0.0
Penco,0,0,1,0,0.0
San Pedro de la Paz,159,56,19,101,94.3
Talcahuano,47,6,3,27,96.4
Tomé,1,1,0,9,100.0


In [107]:
# 5.7  Direcciones sin coordenadas
missing = processed[processed["lat"].isna()]
missing[["commune", "location", "neighborhood"]].drop_duplicates().head(20)

,commune,location,neighborhood
17,Chiguayante,"CONDOMINIO LAS AZUCENAS, Chiguayante",CONDOMINIO LAS AZUCENAS
36,Concepción,"Reserva Nonguen, Concepción",Reserva Nonguen
42,Concepción,"Av. Roosevelt 1666, Concepción, Concepción",Concepción
110,Coronel,"Villa Esperanza, Coronel",Villa Esperanza
121,Coronel,"Pje. Frutillar, Coronel, Coronel",Coronel
122,Hualpén,"Toulouse, Hualpén, Hualpén",Hualpén
123,Hualpén,"Pje. Arezzo, Hualpén, Hualpén",Hualpén
124,Hualqui,"HUALQUI, Hualqui",HUALQUI
125,Hualqui,Hualqui,NaN
126,Penco,"Condominio Terrazas De Landa, Penco, Penco",Penco


## 6. Mapa de verificación
Solo para comprobar que los puntos caen donde deben.

In [108]:
points = processed.dropna(subset=["lat"]).copy()

# Readable text for the tooltip
points["price_text"] = points.apply(
    lambda r: f"${r['price_clp']:,.0f}".replace(",", ".")
    + (f"  (UF {r['price']:,.1f})".replace(".", ",") if r["currency"] == "UF" else ""),
    axis=1,
)
points["m2_text"] = points["usable_m2"].map(lambda v: f"{v:.0f} m²" if pd.notna(v) else "sin dato")
points["bed_text"] = points["bedrooms"].map(lambda v: f"{v:.0f} dorm." if pd.notna(v) else "")
points["bath_text"] = points["bathrooms"].map(lambda v: f"{v:.0f} baños" if pd.notna(v) else "")
points["neighborhood_text"] = points["neighborhood"].fillna("")

fig = px.scatter_map(
    points, lat="lat", lon="lon", color="commune",
    custom_data=["title", "price_text", "property_type", "bed_text", "bath_text",
                 "m2_text", "neighborhood_text", "commune"],
    zoom=10, height=650,
    labels={"commune": "Comuna"},
)

fig.update_traces(
    marker=dict(size=6, opacity=0.75),
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>"
        "%{customdata[0]}<br>"
        "<span style='color:#666'>%{customdata[2]} · %{customdata[3]} · "
        "%{customdata[4]} · %{customdata[5]}</span><br>"
        "<span style='color:#666'>%{customdata[6]}, %{customdata[7]}</span>"
        "<extra></extra>"
    ),
)

fig.update_layout(
    hoverlabel=dict(bgcolor="white", font_size=13, font_family="Arial",
                    bordercolor="#ccc"),
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(bgcolor="rgba(255,255,255,0.85)", x=0.01, y=0.99),
)
fig.show()

In [ ]:

pts = processed.dropna(subset=["lat"]).copy()

# Center of each commune = median of its points (robust to a few wrong ones)
centers = pts.groupby("commune")[["lat", "lon"]].median()
pts = pts.join(centers, on="commune", rsuffix="_center")

# Approximate distance in km (good enough at this scale)
pts["km_from_center"] = np.sqrt(
    ((pts["lat"] - pts["lat_center"]) * 111) ** 2
    + ((pts["lon"] - pts["lon_center"]) * 111 * np.cos(np.radians(pts["lat"]))) ** 2
).round(1)

(pts.sort_values("km_from_center", ascending=False)
    [["commune", "location", "precision", "km_from_center", "lat", "lon"]]
    .head(20))

,commune,location,precision,km_from_center,lat,lon
117,Coronel,"Valle Mitrinhue, Coronel, Coronel",street,15.7,-36.980118,-72.967104
119,Coronel,"Valle Mitrinhue, Coronel, Coronel",street,15.7,-36.980118,-72.967104
118,Coronel,"COD40008 Valle Mitrinhue, Camino A Santa Juana...",neighborhood,14.5,-37.054672,-72.984795
111,Coronel,"Camino A Santa Juana, Coronel",street,14.5,-37.054672,-72.984795
135,San Pedro de la Paz,"Camino A Santa Juana, San Pedro De La Paz",street,12.3,-36.951426,-73.049250
1624,Tomé,"Av. Werner 2500, Tome, Tomé, Ciudad De Tomé, Tomé",address,9.2,-36.609562,-72.964808
1622,Tomé,"Werner, 2500, Ciudad De Tomé, Tomé",street,8.9,-36.605335,-72.969575
1297,San Pedro de la Paz,"Concepción, Villa San Pedro - Bayona, San Pedr...",street,7.2,-36.905169,-73.144816
1440,San Pedro de la Paz,"Los Mañios 6095, San Pedro De La Paz, Portal S...",address,7.0,-36.903396,-73.144057
1541,San Pedro de la Paz,"Los Mañios 6095, San Pedro De La Paz, Portal S...",address,7.0,-36.903396,-73.144057


In [76]:
processed.groupby(["property_type", "currency"])["price"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).round(1)

count       mean       std       min        1%        5%       50%        95%        99%        max
property_type currency                                                                                                      
casa          CLP        200.0  1197250.0  775520.2  350000.0  350000.0  460000.0  990000.0  2800000.0  3203000.0  7000000.0
              UF          23.0      390.1    1571.8      22.0      22.4      26.1      61.0      117.5     5954.4     7600.0
departamento  CLP       1377.0   552799.6  185637.8  190000.0  320000.0  360000.0  510000.0   850000.0  1300000.0  1950000.0
              UF          25.0      306.9    1026.7       8.0       8.7      11.2      19.0     2247.0     4054.0     4450.0

In [85]:
suspicious = processed[
    ((processed["currency"] == "UF") & (processed["price"] > 100))
    | ((processed["currency"] == "CLP") & (processed["price"] > 4_000_000))
    | ((processed["currency"] == "CLP") & (processed["price"] < 250_000))
]

(suspicious[["property_type", "commune", "price", "currency",
             "bedrooms", "usable_m2", "title", "url"]]
    .sort_values(["currency", "price"]))

,property_type,commune,price,currency,bedrooms,usable_m2,title,url
945,departamento,Concepción,190000.0,CLP,1.0,30.0,"Depto Arriendo 1d1b, Heras Con Janequeo. Prime...",https://portalinmobiliario.com/MLC-2340710863-...
784,departamento,Concepción,225000.0,CLP,2.0,42.0,Depto. 2d2b En Concepcion,https://portalinmobiliario.com/MLC-4619918308-...
307,departamento,Concepción,245000.0,CLP,1.0,24.0,Departamento Vicuña Mackenna Id: 45997,https://portalinmobiliario.com/MLC-4457873246-...
38,casa,Concepción,7000000.0,CLP,20.0,395.0,Casa Remodelada 16 Estacionamientos,https://portalinmobiliario.com/MLC-4479042762-...
24,casa,Concepción,120.0,UF,3.0,NaN,Plaza Juan Bosco,https://portalinmobiliario.com/MLC-2312847655-...
1013,departamento,Concepción,2800.0,UF,2.0,55.0,"Central, A Pasos De Paicaví Y Uss, 2 Dorm Mari...",https://portalinmobiliario.com/MLC-2256962597-...
345,departamento,Concepción,4450.0,UF,2.0,55.0,Departamento Con Vista Al Rio Bio Bio,https://portalinmobiliario.com/MLC-2354670487-...
44,casa,Concepción,7600.0,UF,5.0,142.0,Excelente Casa Condominio Lomas De San Andrés,https://portalinmobiliario.com/MLC-4503070128-...
